# Module 6 Exercise: nanoGPT-style model on tiny Shakespeare

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nsteve2407/llm-transformers-course/blob/master/notebooks/06-scaling-modern-llms/exercise_starter.ipynb)

Module page: [Module 6: Scaling & Modern LLMs](https://nsteve2407.github.io/llm-transformers-course/modules/06-scaling-modern-llms/)

A from-scratch, self-contained decoder-only Transformer ("nanoGPT-style"), trained as a character-level
language model on the tiny Shakespeare corpus. No `transformers` dependency -- everything here (tokenizer,
attention, MLP, training loop, sampling) is built from raw `torch`/`torch.nn` ops, in the same
manual-implementation spirit as `01-dnn-refresher` and Module 4's from-scratch attention.

**Part A**: data loading (tiny Shakespeare, or synthetic text under `SMOKE_TEST`) and a character-level tokenizer.

**Part B**: causal multi-head self-attention, built and verified from scratch (no `nn.MultiheadAttention`).
**You will implement `CausalSelfAttention.forward`.**

**Part C**: two interchangeable MLP blocks -- the baseline GELU MLP, and a from-scratch **SwiGLU** MLP (this
notebook's chosen "modern LLM" component) whose hidden dimension is solved for so both MLPs have essentially
the same parameter count. **You will implement `GELUMlp.forward` and `SwiGLUMlp.forward`.**

**Part D**: the decoder block (pre-norm, causal self-attn + MLP) and the full GPT model (token + learned
positional embeddings, a stack of blocks, final LayerNorm, weight-tied output head).

**Part E**: the training loop -- random `(context, target)` batch sampling, AdamW, cross-entropy next-token
loss, periodic train/val loss estimation, loss-curve plot -- run once for the baseline model and once for
the SwiGLU model, at matched parameter count and identical training budget.

**Part F**: autoregressive sampling with temperature and top-k filtering, compared side-by-side between the
two models.

**Part G**: discussion -- did SwiGLU measurably help at this toy scale, and how does that relate to why it's
used in LLaMA-scale models?

In [ ]:
import os
import math
import random
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
torch.manual_seed(0)
random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"SMOKE_TEST={SMOKE_TEST}, device={device}")

## Data, hyperparameters, and design choices (documented judgment calls)

- **`SMOKE_TEST=1`** skips the network entirely: instead of downloading tiny Shakespeare, we generate a
  synthetic character corpus from a small fixed 20-character vocabulary. It isn't pure noise, though --
  pure i.i.d. random characters carry no learnable structure at all, so a tiny model's loss would sit flat
  at `ln(vocab_size)` and never move. Instead we repeat a small fixed set of random "words" from that
  vocabulary in random order, which gives even a 2-layer, 32-dim model something real to memorize within a
  couple hundred optimizer steps, so the `SMOKE_TEST` run still exercises "does the loss actually go down."
- Full-scale model: `n_embd=256, n_head=4, n_layer=6, block_size=256` -- with weight tying between the
  token-embedding table and the output head (standard GPT-2/nanoGPT practice: the same matrix maps
  characters to vectors and vectors back to character logits), this comes to ~4.8M parameters, comfortably
  inside the 1-10M target range from the exercise brief.
- `SMOKE_TEST` model: `n_embd=32, n_head=4, n_layer=2, block_size=32` -- tiny, but every architectural piece
  (multi-head causal attention, two-block depth, both MLP variants) is still exercised.
- **GPT-2-style weight init** (`N(0, 0.02)` for every `nn.Linear`/`nn.Embedding` weight, zero bias) is used
  throughout, rather than PyTorch's default `nn.Linear` init. This matters here: with the default init and
  weight tying, the untrained logits come out unusually large and the initial loss can land far from the
  sanity-check value `ln(vocab_size)` (empirically, default init gave initial losses around 20+ instead of
  the expected ~2.9-4.2). The GPT-2-style init keeps the untrained model's loss close to the "uniform
  distribution over the vocabulary" baseline, which is the correctness sanity check the training loop below
  relies on.
- **No dropout anywhere.** As in Module 4, this keeps the baseline-vs-SwiGLU comparison a clean,
  deterministic isolation of "which MLP is used" rather than adding another source of run-to-run noise on
  top of an already tiny amount of training data/steps.
- **Weight-tied output head, no bias on the LM head.** Standard nanoGPT/GPT-2 practice; also means the
  baseline and SwiGLU models differ *only* in their MLP block, which is exactly the variable we want to
  isolate (the same "one changed component, everything else held fixed" principle used for PlainNet vs.
  ResNet in Module 2).

In [ ]:
if SMOKE_TEST:
    N_EMBD, N_HEAD, N_LAYER, BLOCK_SIZE = 32, 4, 2, 32
    BATCH_SIZE = 16
    MAX_ITERS = 200
    EVAL_INTERVAL = 40
    EVAL_ITERS = 10
    LR = 1e-2
else:
    N_EMBD, N_HEAD, N_LAYER, BLOCK_SIZE = 256, 4, 6, 256
    BATCH_SIZE = 64
    MAX_ITERS = 3000
    EVAL_INTERVAL = 300
    EVAL_ITERS = 100
    LR = 3e-4

assert N_EMBD % N_HEAD == 0, "n_embd must be divisible by n_head"
print(f"N_EMBD={N_EMBD}, N_HEAD={N_HEAD}, N_LAYER={N_LAYER}, BLOCK_SIZE={BLOCK_SIZE}, "
      f"BATCH_SIZE={BATCH_SIZE}, MAX_ITERS={MAX_ITERS}")

## Part A: data loading and character-level tokenizer

Under `SMOKE_TEST`, a synthetic corpus (fixed 20-char vocabulary, repeated random "words") replaces the
tiny Shakespeare download -- no network access is attempted in that branch. Otherwise we download the
classic ~1.1MB tiny Shakespeare corpus (Karpathy's `char-rnn` repo) and build a character-level vocabulary
from its sorted unique characters (65 characters for the real corpus).

The tokenizer itself is intentionally simple: two dicts, `stoi` (char -> int) and `itos` (int -> char), built
from `sorted(set(text))` so the mapping is deterministic given the corpus.

In [ ]:
TINY_SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

if SMOKE_TEST:
    # Synthetic corpus: no network access. A small fixed 20-character vocabulary, but built from a small
    # set of repeated "words" (not pure i.i.d. noise) so there is learnable local structure -- see the
    # judgment-calls cell above for why pure noise wouldn't let the loss curve actually decrease.
    smoke_vocab_chars = list("abcdefghijklmnopqrs ")  # 20 characters, including space as a separator
    assert len(smoke_vocab_chars) == 20
    smoke_words = ["".join(random.choice(smoke_vocab_chars[:-1]) for _ in range(random.randint(3, 6)))
                   for _ in range(10)]
    text = " ".join(random.choice(smoke_words) for _ in range(4000))
    print(f"SMOKE_TEST synthetic corpus: {len(text)} characters from a fixed 20-char vocabulary, "
          f"built from {len(smoke_words)} repeated words (no download).")
else:
    with urllib.request.urlopen(TINY_SHAKESPEARE_URL) as response:
        text = response.read().decode("utf-8")
    print(f"Downloaded tiny Shakespeare: {len(text)} characters.")

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}


def encode(s):
    return [stoi[c] for c in s]


def decode(ids):
    return "".join(itos[i] for i in ids)


print(f"vocab_size={vocab_size}")
print("vocabulary:", "".join(chars))

In [ ]:
# Sanity check: encode/decode round-trips exactly.
sample = text[:200]
assert decode(encode(sample)) == sample, "tokenizer round-trip failed"
print("encode/decode round-trip: OK")
print("first 120 characters of the corpus:")
print(repr(text[:120]))

data = torch.tensor(encode(text), dtype=torch.long)
n_train = int(0.9 * len(data))
train_data, val_data = data[:n_train], data[n_train:]
print(f"train_data: {len(train_data)} tokens, val_data: {len(val_data)} tokens")

## Part B: causal multi-head self-attention (from scratch)

`CausalSelfAttention` follows the same shape conventions as Module 4's `MultiHeadAttention` (separate
learned `W_q`/`W_k`/`W_v`/`W_o` projections, reshape into `(batch, heads, seq, d_k)`), specialized to
**self-attention only** (query = key = value = the same input) with a fixed **causal mask** baked in: query
position `i` may only attend to key positions `j <= i`. The mask is precomputed once as a `(block_size,
block_size)` buffer of `0`/`-inf` (`register_buffer`, so it moves with `.to(device)` but is never trained)
and sliced down to `(T, T)` for whatever sequence length `T <= block_size` is actually passed in.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        self.n_head = n_head
        self.d_k = n_embd // n_head

        self.W_q = nn.Linear(n_embd, n_embd)
        self.W_k = nn.Linear(n_embd, n_embd)
        self.W_v = nn.Linear(n_embd, n_embd)
        self.W_o = nn.Linear(n_embd, n_embd)

        causal_mask = torch.triu(torch.full((block_size, block_size), float("-inf")), diagonal=1)
        self.register_buffer("causal_mask", causal_mask)  # (block_size, block_size), not a learned Parameter

    def forward(self, x):
        """x: (batch, seq_len, n_embd), seq_len <= block_size. Returns (batch, seq_len, n_embd)."""
        # TODO: project x with W_q/W_k/W_v; reshape each from (batch, seq_len, n_embd) to
        # (batch, n_head, seq_len, d_k) via .view(...).transpose(1, 2). Compute
        # scores = Q @ K.transpose(-2, -1) / sqrt(d_k), add self.causal_mask[:seq_len, :seq_len],
        # softmax over the last dim, multiply by V. Reshape back to (batch, seq_len, n_embd) via
        # .transpose(1, 2).contiguous().view(...) and apply W_o.
        raise NotImplementedError("TODO: implement CausalSelfAttention.forward")

In [ ]:
# Sanity check 1: output shape.
attn_demo = CausalSelfAttention(N_EMBD, N_HEAD, BLOCK_SIZE)
x_demo = torch.randn(4, BLOCK_SIZE, N_EMBD)
out_demo = attn_demo(x_demo)
assert out_demo.shape == x_demo.shape
print("CausalSelfAttention output shape:", out_demo.shape)

# Sanity check 2: causal-mask verification, same technique as Module 4 Part I -- editing the input at
# positions after `modify_pos` must leave every output at position <= modify_pos exactly unchanged.
attn_demo.eval()
modify_pos = BLOCK_SIZE // 2
x1 = torch.randn(1, BLOCK_SIZE, N_EMBD)
x2 = x1.clone()
x2[:, modify_pos + 1:, :] = torch.randn_like(x2[:, modify_pos + 1:, :])
with torch.no_grad():
    out1 = attn_demo(x1)
    out2 = attn_demo(x2)
past_diff = (out1[:, :modify_pos + 1] - out2[:, :modify_pos + 1]).abs().max().item()
future_diff = (out1[:, modify_pos + 1:] - out2[:, modify_pos + 1:]).abs().max().item()
assert past_diff == 0.0, f"causal mask is leaking: past positions changed (max diff {past_diff})"
assert future_diff > 1e-4, "test is vacuous: future positions should differ but don't"
print(f"causal mask verification: past positions unaffected (diff={past_diff}), "
      f"future positions do change (diff={future_diff:.3e}). OK")

## Part C: two MLP blocks at matched parameter count -- baseline GELU MLP vs. from-scratch SwiGLU

**Baseline `GELUMlp`**: the standard GPT-2-style position-wise MLP, `Linear(n_embd, 4*n_embd) -> GELU ->
Linear(4*n_embd, n_embd)`, with biases.

**`SwiGLUMlp`** (this notebook's chosen modern-LLM component, used by LLaMA and most subsequent open
models): a *gated* MLP with three projections instead of two -- `down( SiLU(gate(x)) * up(x) )` -- and no
biases (matching the LLaMA convention). `SiLU(x) = x * sigmoid(x)` (a.k.a. "swish"); the elementwise product
with `up(x)` is the "gate": the network learns to scale each hidden unit up or down per input, rather than
just applying a fixed nonlinearity.

**Matching parameter count.** A naive `hidden_dim = 4 * n_embd` for SwiGLU would give it *three*
`n_embd x hidden_dim` matrices vs. the baseline's *two*, i.e. ~1.5x the baseline's parameters -- not a fair
comparison. Instead we solve for the hidden dimension that makes SwiGLU's parameter count match the
baseline's: baseline has (ignoring biases, which are a small correction) `2 * n_embd * 4*n_embd = 8 *
n_embd^2` weight parameters; SwiGLU (no bias) has `3 * n_embd * hidden_dim`. Setting these equal:
`hidden_dim = baseline_params / (3 * n_embd)`, rounded to the nearest integer. This is exactly the
"one changed component, everything else (including capacity) held fixed" principle used for PlainNet vs.
ResNet in Module 2.

In [ ]:
class GELUMlp(nn.Module):
    """Baseline position-wise MLP: Linear -> GELU -> Linear, with biases."""

    def __init__(self, n_embd):
        super().__init__()
        self.fc1 = nn.Linear(n_embd, 4 * n_embd)
        self.fc2 = nn.Linear(4 * n_embd, n_embd)

    def forward(self, x):
        # TODO: return self.fc2(F.gelu(self.fc1(x)))
        raise NotImplementedError("TODO: implement GELUMlp.forward")


class SwiGLUMlp(nn.Module):
    """Gated SiLU MLP (LLaMA-style): down(SiLU(gate(x)) * up(x)), no biases.

    `hidden_dim` is chosen by the caller (see `swiglu_matched_hidden_dim` below) so this has
    roughly the same total parameter count as `GELUMlp`, despite having three projections instead
    of two.
    """

    def __init__(self, n_embd, hidden_dim):
        super().__init__()
        self.w_gate = nn.Linear(n_embd, hidden_dim, bias=False)
        self.w_up = nn.Linear(n_embd, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, n_embd, bias=False)

    def forward(self, x):
        # TODO: gate = F.silu(self.w_gate(x))  (SiLU(x) = x * sigmoid(x), the modern-component piece
        # this exercise is about); return self.w_down(gate * self.w_up(x)).
        raise NotImplementedError("TODO: implement SwiGLUMlp.forward (the SwiGLU modern component)")


def swiglu_matched_hidden_dim(n_embd):
    """Solve for the SwiGLU hidden_dim that gives ~the same parameter count as GELUMlp(n_embd)."""
    baseline_params = sum(p.numel() for p in GELUMlp(n_embd).parameters())
    hidden_dim = round(baseline_params / (3 * n_embd))
    return hidden_dim, baseline_params

In [ ]:
swiglu_hidden_dim, gelu_mlp_params = swiglu_matched_hidden_dim(N_EMBD)
swiglu_mlp_params = sum(p.numel() for p in SwiGLUMlp(N_EMBD, swiglu_hidden_dim).parameters())
rel_diff = abs(swiglu_mlp_params - gelu_mlp_params) / gelu_mlp_params

print(f"n_embd={N_EMBD}: GELUMlp hidden=4*n_embd={4 * N_EMBD}, params={gelu_mlp_params}")
print(f"n_embd={N_EMBD}: SwiGLUMlp hidden={swiglu_hidden_dim}, params={swiglu_mlp_params} "
      f"(relative diff from GELUMlp: {rel_diff:.2%})")
assert rel_diff < 0.02, "SwiGLU hidden_dim should be solved so parameter counts are matched to within ~2%"


## Part D: the decoder block and the full GPT model

`Block` is a standard **pre-norm** decoder block: `x = x + attn(LN1(x))`, then `x = x + mlp(LN2(x))` --
matching the Pre-LN placement from Module 4 Part D (used by GPT-2 onward), which is what makes training deep
decoder-only stacks stable without a LR-warmup-dependent Post-LN setup. `mlp_type` selects which MLP class
the block uses (`"gelu"` or `"swiglu"`), so the *only* thing that differs between the baseline and SwiGLU
models below is this one flag, threaded down from `GPTLanguageModel`.

`GPTLanguageModel` is the full model: a learned token embedding, a learned positional embedding (added, not
concatenated), a stack of `n_layer` `Block`s, a final LayerNorm, and a linear head back to vocabulary
logits. The head's weight is **tied** to the token embedding's weight (standard GPT-2/nanoGPT practice --
saves `vocab_size * n_embd` parameters and is a mild regularizer). Weights are initialized GPT-2-style
(`N(0, 0.02)`, zero bias) for the reason given in the judgment-calls cell above.

In [ ]:
class Block(nn.Module):
    """Pre-norm decoder block: x = x + attn(LN1(x)); x = x + mlp(LN2(x))."""

    def __init__(self, n_embd, n_head, block_size, mlp_type="gelu", swiglu_hidden_dim=None):
        super().__init__()
        assert mlp_type in ("gelu", "swiglu")
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size)
        self.ln2 = nn.LayerNorm(n_embd)
        if mlp_type == "gelu":
            self.mlp = GELUMlp(n_embd)
        else:
            self.mlp = SwiGLUMlp(n_embd, swiglu_hidden_dim)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, mlp_type="gelu"):
        super().__init__()
        self.block_size = block_size
        self.tok_embedding = nn.Embedding(vocab_size, n_embd)
        self.pos_embedding = nn.Embedding(block_size, n_embd)

        swiglu_hidden_dim = None
        if mlp_type == "swiglu":
            swiglu_hidden_dim, _ = swiglu_matched_hidden_dim(n_embd)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, block_size, mlp_type=mlp_type, swiglu_hidden_dim=swiglu_hidden_dim)
            for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_embedding.weight  # weight tying

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        """idx: (batch, seq_len) of token ids, seq_len <= block_size.
        targets: optional (batch, seq_len) of next-token ids. Returns (logits, loss_or_None).
        """
        batch_size, seq_len = idx.shape
        assert seq_len <= self.block_size, f"sequence length {seq_len} exceeds block_size {self.block_size}"

        positions = torch.arange(seq_len, device=idx.device)
        x = self.tok_embedding(idx) + self.pos_embedding(positions)[None, :, :]
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)  # (batch, seq_len, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Autoregressive sampling. idx: (batch, seq_len) seed context.
        temperature: > 0, scales logits before softmax (lower = more confident/greedy).
        top_k: if set, only sample from the top_k highest-probability tokens at each step.
        Returns (batch, seq_len + max_new_tokens).
        """
        was_training = self.training
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature  # (batch, vocab_size), last-timestep logits
            if top_k is not None:
                top_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                threshold = top_values[:, [-1]]
                logits = logits.masked_fill(logits < threshold, float("-inf"))
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        if was_training:
            self.train()
        return idx

In [ ]:
torch.manual_seed(0)
model_baseline = GPTLanguageModel(vocab_size, N_EMBD, N_HEAD, N_LAYER, BLOCK_SIZE, mlp_type="gelu").to(device)
torch.manual_seed(0)
model_swiglu = GPTLanguageModel(vocab_size, N_EMBD, N_HEAD, N_LAYER, BLOCK_SIZE, mlp_type="swiglu").to(device)

params_baseline = sum(p.numel() for p in model_baseline.parameters())
params_swiglu = sum(p.numel() for p in model_swiglu.parameters())
rel_diff_total = abs(params_swiglu - params_baseline) / params_baseline

print(f"baseline (GELU MLP) model: {params_baseline:,} parameters")
print(f"SwiGLU model:               {params_swiglu:,} parameters (relative diff: {rel_diff_total:.2%})")
assert rel_diff_total < 0.02, "baseline and SwiGLU models should be at matched parameter count"

## Part E: training loop

`get_batch` samples a batch of random `(context, target)` windows of length `BLOCK_SIZE` from either split
(the standard language-model training setup: `target` is `context` shifted one position to the right, i.e.
next-token prediction at every position simultaneously). `estimate_loss` averages the cross-entropy loss
over `EVAL_ITERS` fresh batches from each split (cheaper/less noisy than tracking the single-batch training
loss). `train_model` runs the whole loop -- AdamW, periodic evaluation/printing -- for one model.

We train `model_baseline` and `model_swiglu` with the **exact same** `get_batch`/`train_model` code, same
`MAX_ITERS`, same optimizer settings, and the same random seed for data sampling (`torch.manual_seed` before
each call) -- the only difference between the two runs is which model is being trained.

In [ ]:
def get_batch(split):
    data_split = train_data if split == "train" else val_data
    ix = torch.randint(len(data_split) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([data_split[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([data_split[i + 1:i + BLOCK_SIZE + 1] for i in ix])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_loss(model):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def train_model(model, name, data_seed=0):
    torch.manual_seed(data_seed)  # same batch-sampling sequence for every model we train
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    history = []  # list of (iter, train_loss, val_loss)
    for it in range(MAX_ITERS):
        if it % EVAL_INTERVAL == 0 or it == MAX_ITERS - 1:
            losses = estimate_loss(model)
            history.append((it, losses["train"], losses["val"]))
            print(f"[{name}] iter {it:5d}  train_loss={losses['train']:.4f}  val_loss={losses['val']:.4f}")
        xb, yb = get_batch("train")
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    return history

In [ ]:
print("Training baseline (GELU MLP) model...")
history_baseline = train_model(model_baseline, "baseline")

print("\nTraining SwiGLU model...")
history_swiglu = train_model(model_swiglu, "swiglu")

final_train_baseline = history_baseline[-1][1]
final_train_swiglu = history_swiglu[-1][1]
assert final_train_baseline < history_baseline[0][1], "baseline training loss did not decrease"
assert final_train_swiglu < history_swiglu[0][1], "SwiGLU training loss did not decrease"
print("\nBoth models' training loss decreased from their first to last evaluation: OK")

In [ ]:
plt.figure()
iters_b, train_b, val_b = zip(*history_baseline)
iters_s, train_s, val_s = zip(*history_swiglu)
plt.plot(iters_b, train_b, label="baseline train", color="tab:blue")
plt.plot(iters_b, val_b, label="baseline val", color="tab:blue", linestyle="--")
plt.plot(iters_s, train_s, label="SwiGLU train", color="tab:orange")
plt.plot(iters_s, val_s, label="SwiGLU val", color="tab:orange", linestyle="--")
plt.xlabel("iteration")
plt.ylabel("cross-entropy loss")
plt.legend()
plt.title("Training loss: baseline (GELU MLP) vs. SwiGLU, matched parameter count")
plt.show()

## Part F: autoregressive sampling

`generate` starts from a seed context (here, a single "start of sequence" token -- character id `0`) and
repeatedly: crops the context to the last `BLOCK_SIZE` tokens (the model has no memory beyond that), runs a
forward pass, takes the logits at the last timestep, scales by `temperature` (lower = more confident/greedy,
higher = more random), optionally restricts to the `top_k` highest-probability tokens (zeroing out the
rest), samples one token from the resulting distribution, and appends it. This is exactly how GPT-style
models produce open-ended text at inference time.

In [ ]:
torch.manual_seed(42)
seed_context = torch.zeros((1, 1), dtype=torch.long, device=device)  # single start token
gen_len = 80 if SMOKE_TEST else 300

baseline_sample = model_baseline.generate(seed_context, gen_len, temperature=0.8, top_k=20)
swiglu_sample = model_swiglu.generate(seed_context, gen_len, temperature=0.8, top_k=20)

print(f"generated length: {baseline_sample.shape[1]} tokens (requested {gen_len} + 1 seed token)")
assert baseline_sample.shape[1] == gen_len + 1
assert swiglu_sample.shape[1] == gen_len + 1

print("\n--- baseline (GELU MLP) sample ---")
print(decode(baseline_sample[0].tolist()))
print("\n--- SwiGLU sample ---")
print(decode(swiglu_sample[0].tolist()))

In [ ]:
# Concrete check that top_k actually constrains sampling: top_k=1 is effectively greedy decoding
# (only one token has nonzero probability at each step), so the output must be identical regardless
# of the random seed used for torch.multinomial.
torch.manual_seed(1)
greedy1 = model_baseline.generate(seed_context, 30, temperature=1.0, top_k=1)
torch.manual_seed(999)
greedy2 = model_baseline.generate(seed_context, 30, temperature=1.0, top_k=1)
assert torch.equal(greedy1, greedy2), "top_k=1 should be deterministic regardless of random seed"
print("top_k=1 (greedy) sampling is deterministic across different random seeds: OK")
print("greedy sample:", repr(decode(greedy1[0].tolist())))

In [ ]:
# Illustration that temperature changes how "peaky" vs. "random" sampling is.
torch.manual_seed(0)
low_temp_sample = model_baseline.generate(seed_context, 40, temperature=0.2, top_k=None)
torch.manual_seed(0)
high_temp_sample = model_baseline.generate(seed_context, 40, temperature=1.5, top_k=None)
print("low temperature (0.2, near-greedy): ", repr(decode(low_temp_sample[0].tolist())))
print("high temperature (1.5, more random):", repr(decode(high_temp_sample[0].tolist())))

## Part G: did SwiGLU measurably help at this toy scale?

Compare the final train/val losses and the generated samples above. At this tiny scale (small model, tiny
corpus/`BLOCK_SIZE`, at most a few thousand optimizer steps), don't expect a dramatic, unambiguous win for
SwiGLU over the baseline GELU MLP -- and if the numbers above come out close (or even slightly favor the
baseline on a given run/seed), that is itself the expected and informative result, not a sign anything is
broken.

**Why SwiGLU is used at LLaMA scale, and why that motivation doesn't fully show up here:**
- SwiGLU's real advantage is about *expressive capacity per parameter* -- the gating mechanism lets the MLP
  learn input-dependent, multiplicative feature selection instead of a fixed nonlinearity applied uniformly,
  which pays off in the underfitting regime typical of large models trained on huge, diverse corpora. A
  ~1-5M-parameter model trained on tens of thousands of characters of repetitive/synthetic text (or even the
  full ~1MB tiny Shakespeare corpus) is nowhere near that regime -- it's much more likely to be bottlenecked
  by data and training budget than by MLP expressivity, so a meaningfully different loss curve isn't
  expected.
- The comparison here *is* still a fair one (matched parameter count, matched training budget, everything
  else held fixed) -- what's toy-scale is the *effect size* you'd expect to observe, not the experimental
  design.
- (For context, if this notebook had instead implemented **RoPE**, the same gap would show up even more
  starkly: RoPE's main practical benefit is extrapolating to sequence lengths *longer than what was trained
  on*, or encoding *relative* position more gracefully at long range -- neither of which a fixed
  `block_size=32` toy run ever exercises. Likewise **RMSNorm**'s benefit over LayerNorm is primarily
  *compute* efficiency (skipping the mean-subtraction reduction), which doesn't show up in a loss-curve
  comparison at all -- you'd have to measure wall-clock time or FLOPs, not loss.)
- The broader lesson: a component's motivation at frontier scale ("why this exists") and what a small toy
  experiment can actually demonstrate ("what you'd observe here") are often different questions, and it's
  worth being explicit about which one an experiment is actually answering.

## Summary

Implemented from scratch and trained end-to-end:
- **Character-level tokenizer** (`stoi`/`itos` dicts) over either the real tiny Shakespeare corpus or a
  synthetic `SMOKE_TEST` corpus (no network access in that branch).
- **`CausalSelfAttention`**: multi-head self-attention with a precomputed additive causal mask, verified both
  by shape and by a concrete "editing a future token doesn't change earlier outputs" proof (Module 4's Part
  I technique, reapplied here).
- **Two MLP blocks at matched parameter count**: baseline `GELUMlp` (GPT-2-style) and a from-scratch
  `SwiGLUMlp` (LLaMA-style gated SiLU MLP), with the SwiGLU hidden dimension explicitly solved for so the two
  are a fair, isolated-variable comparison.
- **`Block`/`GPTLanguageModel`**: a pre-norm decoder-only Transformer (token + learned positional embeddings,
  a stack of blocks, final LayerNorm, weight-tied output head), ~4.8M parameters at full scale.
- **Training loop**: random-batch sampling, AdamW, cross-entropy loss, periodic train/val evaluation, a
  loss-curve plot -- run identically for the baseline and SwiGLU models.
- **Autoregressive sampling** with temperature and top-k filtering, including a concrete check that `top_k=1`
  sampling is deterministic (true greedy decoding) and a temperature comparison.
- **Discussion** of why a toy-scale comparison doesn't (and isn't expected to) reproduce the full motivation
  for SwiGLU/RoPE/RMSNorm at frontier scale.